# R-MOVE · 01 · ¿Son fiables los datos de movilidad de Telefónica?

## Cómo leer este notebook

**Objetivo.** Comprobar la calidad de los tres conjuntos de datos de Telefónica Smart Steps (enero de 2023 – agosto de 2025) antes de analizarlos, y preparar el fichero de movilidad entre municipios que usan los notebooks siguientes.

| Conjunto | Qué contiene | Fichero mensual | ¿Se usa después? |
|---|---|---|---|
| Movilidad entre barrios | Viajes de un barrio a otro dentro de Barcelona | `movilidad_barrios_AAAA-MM_origen.csv` | No: solo se valida |
| Movilidad entre municipios | Viajes de un municipio a otro | `movilidad_municipios_AAAA-MM_origen.csv` | Sí: notebooks 02, 03 y 04 |
| Movilidad municipio–barrio | Viajes entre un municipio y un barrio de Barcelona, en los dos sentidos | `movilidad_mun_barrios_AAAA-MM_origen.csv` | Sí: notebook 04 |

**Entrada:** `data/raw/telefonica/AAAA-MM/` (32 meses).
**Salida:** `data/processed/telefonica/movilidad_municipios.parquet` (sección B).

**Resultado principal.**
- Los tres conjuntos son estructuralmente correctos: no hay nulos, fechas inválidas, relaciones geográficas incorrectas ni valores de viajes que incumplan las reglas de Telefónica.
- Faltan los mismos 3 días en los tres conjuntos (11/02/2024, 13/04/2025 y 20/05/2025): cobertura del 99,69 %. No se imputan.
- En la movilidad municipio–barrio, el 27,7 % de las filas están repetidas, casi todas con 0 viajes: eliminarlas solo cambiaría el volumen un ≈ 1 %. Se suman (decisión del notebook 04).

| Sección | Contenido |
|---|---|
| **A. Preparación** | Configuración, carga y funciones de validación |
| **1. Un mes en detalle** | Enero de 2023: columnas, nulos, filas repetidas, fechas, geografía y viajes |
| **2. Todo el periodo** | Las mismas comprobaciones en los 32 meses y días sin datos |
| **3. Filas repetidas** | Qué son, cuánto pesan en el volumen y qué se decide |
| **4. Conclusiones** | |
| **B. Exportación** | `movilidad_municipios.parquet` |

# A. Preparación

### A.1 Configuración

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
RAW = ROOT / "data" / "raw" / "telefonica"
SALIDA = ROOT / "data" / "processed" / "telefonica" / "movilidad_municipios.parquet"
BCN = "08019"

# Los tres conjuntos de Telefónica y el nombre de su fichero mensual
CONJUNTOS = {
    "barrios": "movilidad_barrios_{mes}_origen.csv",
    "municipios": "movilidad_municipios_{mes}_origen.csv",
    "municipio_barrio": "movilidad_mun_barrios_{mes}_origen.csv",
}
# Los códigos geográficos se leen como texto para no perder los ceros iniciales (08019)
TIPOS_GEO = {c: "string" for c in ["municipio_origen", "municipio_destino", "barrio_origen", "barrio_destino"]}

MESES = sorted(p.name for p in RAW.iterdir() if p.is_dir())
print(f"{len(MESES)} meses: {MESES[0]} – {MESES[-1]}")

### A.2 Carga y funciones de validación

Cada función devuelve un diccionario con el número de problemas encontrados (0 = correcto), para poder juntar después los resultados de todos los meses en una sola tabla.

In [ ]:
def cargar(mes, conjunto):
    """Lee el CSV mensual de un conjunto y convierte la fecha."""
    df = pd.read_csv(RAW / mes / CONJUNTOS[conjunto].format(mes=mes), dtype=TIPOS_GEO)
    df["day"] = pd.to_datetime(df["day"], errors="coerce")
    return df


def validar_calidad(df):
    """Filas, valores nulos y filas repetidas exactamente."""
    return {"Registros": len(df), "Nulos": int(df.isna().sum().sum()), "Duplicados": int(df.duplicated().sum())}


def validar_temporal(df, mes):
    """Días que faltan en el mes y coherencia de la fecha con el día de la semana y el mes."""
    esperadas = pd.date_range(f"{mes}-01", pd.Timestamp(f"{mes}-01") + pd.offsets.MonthEnd(0))
    presentes = pd.DatetimeIndex(df["day"].dropna().unique())
    dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
    dia_real = df["day"].dt.day_of_week.map(dict(enumerate(dias)))
    return {
        "Días esperados": len(esperadas),
        "Días presentes": len(presentes.intersection(esperadas)),
        "Días faltantes": len(esperadas.difference(presentes)),
        "Fechas inválidas": int(df["day"].isna().sum()),
        "Fechas fuera del mes": int((df["day"].notna() & ~df["day"].between(esperadas.min(), esperadas.max())).sum()),
        "Errores día semana": int((dia_real != df["day_of_week"].str.strip().str.title()).sum()),
        "Errores mes": int((df["day"].dt.month != df["month"]).sum()),
    }


def validar_geografia(df, conjunto):
    """Relaciones origen–destino permitidas, códigos de barrio y un único nombre por código."""
    o_bcn, d_bcn = df["municipio_origen"].eq(BCN), df["municipio_destino"].eq(BCN)
    if conjunto == "barrios":              # origen y destino en Barcelona, siempre con barrio
        relaciones = ~(o_bcn & d_bcn)
        barrios = df["barrio_origen"].eq("-") | df["barrio_destino"].eq("-")
    elif conjunto == "municipios":         # de un municipio a otro distinto, sin barrios
        relaciones = df["municipio_origen"].eq(df["municipio_destino"])
        barrios = pd.Series(False, index=df.index)
    elif conjunto == "municipio_barrio":   # exactamente un extremo en Barcelona, y solo ese con barrio
        relaciones = o_bcn.eq(d_bcn)
        barrios = o_bcn.eq(df["barrio_origen"].eq("-")) | d_bcn.eq(df["barrio_destino"].eq("-"))
    else:
        raise ValueError(f"Conjunto desconocido: {conjunto}")

    columnas = [("municipio_origen", "municipio_origen_name"), ("municipio_destino", "municipio_destino_name")]
    if conjunto != "municipios":
        columnas += [("barrio_origen", "barrio_origen_name"), ("barrio_destino", "barrio_destino_name")]
    nombres = sum(int((df.groupby(c)[n].nunique() > 1).sum()) for c, n in columnas)
    return {"Errores relaciones": int(relaciones.sum()), "Errores barrios": int(barrios.sum()),
            "Inconsistencias código-nombre": nombres}


def validar_viajes(df):
    """Telefónica publica 0 o 15 viajes o más: no puede haber negativos ni valores entre 1 y 14."""
    v = df["viajes"]
    return {"Viajes negativos": int((v < 0).sum()), "Viajes entre 1 y 14": int(v.between(1, 14).sum())}


def validar(df, mes, conjunto):
    """Todas las comprobaciones de un conjunto en un mes."""
    return {**validar_calidad(df), **validar_temporal(df, mes),
            **validar_geografia(df, conjunto), **validar_viajes(df)}

# Resultados

## 1. Un mes en detalle: enero de 2023

Antes de recorrer los 32 meses, se revisa a fondo un mes: qué columnas tiene cada conjunto y si cumple las reglas de Telefónica.

In [ ]:
MES = "2023-01"
datos = {c: cargar(MES, c) for c in CONJUNTOS}
for c, df in datos.items():
    print(f"{c:<17} {len(df):>10,} filas · {df.shape[1]} columnas")

### 1.1 Columnas

In [ ]:
pd.DataFrame({c: df.dtypes.astype(str) for c, df in datos.items()}).fillna("—")

**Resultado.** Los tres conjuntos comparten la fecha (`day`, ya convertida a fecha al cargarla), el día de la semana, el mes, el segmento de población (`origen`: Regional, Residente…), los municipios de origen y destino con su nombre y el número de `viajes` (entero). Los dos conjuntos con barrios añaden el barrio de origen y el de destino.

### 1.2 Nulos y filas repetidas

In [ ]:
pd.DataFrame({c: validar_calidad(df) for c, df in datos.items()}).T

**Resultado.** No hay nulos en ningún conjunto. En la movilidad municipio–barrio hay **728.722 filas repetidas exactamente** de 2.641.454 (27,6 %). La sección 3 las analiza.

### 1.3 Fechas

In [ ]:
pd.DataFrame({c: validar_temporal(df, MES) for c, df in datos.items()}).T

**Resultado.** Los 31 días de enero están en los tres conjuntos, sin fechas inválidas ni incoherencias con el día de la semana o el mes.

### 1.4 Coherencia geográfica

Cada conjunto tiene sus reglas:
- **Entre barrios:** origen y destino en Barcelona, siempre con barrio.
- **Entre municipios:** origen y destino distintos.
- **Municipio–barrio:** exactamente uno de los dos extremos es Barcelona, y solo ese extremo tiene barrio (el otro vale «-»).

También se comprueba que cada código tenga un único nombre.

In [ ]:
pd.DataFrame({c: validar_geografia(df, c) for c, df in datos.items()}).T

**Resultado.** Ningún error en enero: las relaciones respetan las reglas de Telefónica y cada código tiene un único nombre. Queda pendiente contrastar los códigos con fuentes geográficas oficiales.

### 1.5 Valores de viajes

Según la documentación de Smart Steps, los flujos de menos de 15 viajes se publican como 0 por privacidad. Por tanto, `viajes` solo puede valer 0 o 15 o más: $V \in \{0\} \cup \{15, 16, 17, \ldots\}$.

In [ ]:
pd.DataFrame({c: validar_viajes(df) for c, df in datos.items()}).T

**Resultado.** No hay valores negativos ni entre 1 y 14: se cumple la regla. Los ceros se conservan, pero un 0 puede ser cualquier valor entre 0 y 14, lo que afecta sobre todo a los flujos pequeños.

## 2. Todo el periodo (enero de 2023 – agosto de 2025)

Las mismas comprobaciones, mes a mes, en los tres conjuntos. *Tarda unos 10 minutos: lee los 96 ficheros CSV.*

In [ ]:
filas = []
for mes in MESES:
    for conjunto in CONJUNTOS:
        filas.append({"Mes": mes, "Conjunto": conjunto, **validar(cargar(mes, conjunto), mes, conjunto)})
    print(mes, end=" · ", flush=True)
calidad = pd.DataFrame(filas)

resumen = calidad.groupby("Conjunto", sort=False).sum(numeric_only=True)
resumen["% Duplicados"] = resumen["Duplicados"] / resumen["Registros"] * 100
resumen = resumen.drop(columns=["Días esperados", "Días presentes"])
resumen.style.format("{:,.0f}").format({"% Duplicados": "{:.2f} %"})

**Resultado.**
- **Estructura correcta en todo el periodo:** ningún nulo, fecha inválida, relación geográfica incorrecta ni valor de viajes fuera de las reglas, en ninguno de los tres conjuntos.
- **Filas repetidas:** solo en la movilidad municipio–barrio, y en todos los meses: 23.790.018 de 85.823.522 (27,72 %).
- **Faltan 3 días** en cada conjunto (sección 2.1).

### 2.1 Días sin datos

In [ ]:
faltan = {}
for mes in calidad.loc[calidad["Días faltantes"] > 0, "Mes"].unique():
    esperadas = pd.date_range(f"{mes}-01", pd.Timestamp(f"{mes}-01") + pd.offsets.MonthEnd(0))
    for conjunto in CONJUNTOS:
        presentes = pd.DatetimeIndex(cargar(mes, conjunto)["day"].dropna().unique())
        faltan[(mes, conjunto)] = ", ".join(esperadas.difference(presentes).strftime("%d/%m/%Y"))
display(pd.Series(faltan, name="Días sin datos").unstack())

mun = calidad[calidad.Conjunto == "municipios"]
total, sin_datos = mun["Días esperados"].sum(), mun["Días faltantes"].sum()
print(f"Cobertura: {total - sin_datos} de {total} días ({(total - sin_datos) / total:.2%})")

**Resultado.** Faltan los mismos tres días en los tres conjuntos: **11/02/2024, 13/04/2025 y 20/05/2025**. La cobertura es del 99,69 % (971 de 974 días). No se imputan: los análisis posteriores trabajan solo con los días que tienen datos.

## 3. Filas repetidas en la movilidad municipio–barrio

En enero de 2023, el 27,6 % de las filas aparecen repetidas exactamente: mismo día, segmento, municipio, barrio y número de viajes. La documentación de Telefónica no lo explica. Se analiza qué filas son y cuánto pesan en el volumen de viajes.

### 3.1 ¿Qué filas se repiten?

In [ ]:
mb = datos["municipio_barrio"]
rep = mb[mb.duplicated()]
print(f"Filas repetidas: {len(rep):,} · con 0 viajes: {rep.viajes.eq(0).mean():.1%} · "
      f"con viajes > 0: {rep.viajes.gt(0).mean():.1%}")
rep.loc[rep.viajes > 0, "viajes"].describe().round(1).to_frame("Filas repetidas con viajes > 0").T

In [ ]:
frec = mb.value_counts().reset_index(name="repeticiones")
tabla = pd.crosstab(frec["repeticiones"], np.where(frec["viajes"].eq(0), "0 viajes", "Viajes > 0"))
tabla["Total"] = tabla.sum(axis=1)
tabla["% con 0 viajes"] = tabla["0 viajes"] / tabla["Total"] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.bar(tabla.index, tabla["Total"], color="0.6")
ax1.set(title="¿Cuántas veces aparece cada fila? (enero de 2023)", xlabel="Veces que aparece",
        ylabel="Filas distintas (escala log.)", yscale="log")
ax2.bar(tabla.index, tabla["% con 0 viajes"], color="C0")
ax2.set(title="Filas con 0 viajes según las veces que aparecen", xlabel="Veces que aparece",
        ylabel="% con 0 viajes", ylim=(0, 105))
for ax in (ax1, ax2):
    ax.set_xticks(tabla.index)
plt.tight_layout()
plt.show()
tabla.style.format("{:,.0f}").format({"% con 0 viajes": "{:.1f} %"})

**Resultado.** Casi todas las filas repetidas tienen 0 viajes (98,7 %). Cuantas más veces aparece una fila, más probable es que valga 0: son ceros el 63 % de las filas que aparecen una vez, el 95 % de las que aparecen dos y prácticamente el 100 % de las que aparecen tres o más. Las repetidas con viajes son flujos pequeños (mediana de 23 viajes), aunque alguna llega a 980.

### 3.2 ¿Cuánto cambia el volumen si se eliminan?

Análisis de sensibilidad sobre una copia de enero de 2023: reducción del volumen = (viajes con todas las filas − viajes sin las repetidas) / viajes con todas las filas.

In [ ]:
sin_rep = mb.drop_duplicates()
v_todas, v_sin = mb.viajes.sum(), sin_rep.viajes.sum()
print(f"Filas:  {len(mb):,} → {len(sin_rep):,} ({1 - len(sin_rep) / len(mb):.2%} menos)")
print(f"Viajes: {v_todas:,} → {v_sin:,} ({(v_todas - v_sin) / v_todas:.2%} menos)")

**Resultado.** Eliminar las filas repetidas quita el 27,59 % de las filas, pero solo el **0,96 %** de los viajes (de 33.258.586 a 32.939.227), porque casi todas valen 0. El efecto en el volumen total es pequeño, aunque podría ser mayor en algunos pares municipio–barrio concretos.

### 3.3 Decisión

**Las filas repetidas se suman, no se eliminan.** En el notebook 04 (anexo B.2) se ve que una misma combinación de día, segmento, municipio y barrio puede tener hasta 10 filas, a menudo con valores distintos, y más en los municipios grandes (Badalona hasta 9, Sabadell 7). La explicación más probable es que Telefónica divide los municipios grandes en subzonas y solo publica el código del municipio. Con la suma, la movilidad municipio–barrio equivale al ≈ 84–87 % de la movilidad entre municipios, que es razonable; eliminando las repetidas se perdería ≈ 1 % del volumen sin justificación. *Pendiente de confirmar con el mentor de Telefónica.*

## 4. Conclusiones

1. **Los datos son estructuralmente correctos.** En los 32 meses y los tres conjuntos no hay nulos, fechas inválidas, relaciones geográficas incorrectas ni valores de viajes fuera de las reglas de Telefónica.
2. **Faltan 3 días** (11/02/2024, 13/04/2025 y 20/05/2025) en los tres conjuntos: cobertura del 99,69 %. No se imputan.
3. **Un 0 no siempre es un 0:** significa entre 0 y 14 viajes. Afecta a los flujos pequeños; por eso los análisis posteriores se centran en los municipios con ≥ 200 viajes al día hacia Barcelona.
4. **Filas repetidas en la movilidad municipio–barrio (27,7 %):** casi todas valen 0 y pesan ≈ 1 % del volumen. Se suman (notebook 04).
5. **Qué se usa después:** la movilidad entre municipios (notebooks 02, 03 y 04; se exporta en la sección B) y la municipio–barrio (notebook 04). La movilidad entre barrios no se usa porque solo recoge viajes dentro de Barcelona.

# B. Exportación

### B.1 Movilidad entre municipios (`movilidad_municipios.parquet`)

Une los 32 ficheros mensuales en un único Parquet, con la fecha ya convertida y los códigos como texto. Es la entrada de los notebooks 02, 03 y 04. Se escribe mes a mes para no cargar los 65 millones de filas en memoria.

Si el fichero ya existe, solo se comprueba que tiene las mismas filas que los CSV. Para regenerarlo, bórralo y vuelve a ejecutar la celda.

In [ ]:
filas_csv = int(calidad.loc[calidad.Conjunto == "municipios", "Registros"].sum())

if not SALIDA.exists():
    SALIDA.parent.mkdir(parents=True, exist_ok=True)
    escritor = None
    for mes in MESES:
        tabla = pa.Table.from_pandas(cargar(mes, "municipios"), preserve_index=False)
        if escritor is None:
            escritor = pq.ParquetWriter(SALIDA, tabla.schema)
        escritor.write_table(tabla.cast(escritor.schema))
    escritor.close()
    print(f"Guardado: {SALIDA.relative_to(ROOT)}")

filas_parquet = pq.ParquetFile(SALIDA).metadata.num_rows
print(f"{filas_parquet:,} filas en el Parquet · {filas_csv:,} en los CSV · "
      f"{'coinciden' if filas_parquet == filas_csv else 'NO COINCIDEN'}")